# Определение автоматизированного трафика

## Краткий результат

Для каждой куки строится score от 0 до 1 по событиям внутри суточного окна. Используется ансамбль четырёх CatBoost с фиксированным random_seed=42. На платформе пользователь получил **Precision при Recall ≥ 70% = 0,80750**. Это внешний результат по присланному скриншоту, а не вычисление скрытых меток в ноутбуке.

Этот ноутбук запускает обучение из исходных данных и проверяет совпадение полученного CSV с отправленным файлом.

## Метод и ограничения

- Единица наблюдения — cookie_id. Идентификатор сохраняется в ответе, но не подаётся в модель.
- Для признаков учитываются только события в интервале **[window_start_ts, window_end_ts)**.
- Признаки: возраст куки, частоты действий, паузы, разнообразие объявлений и запросов, платформа и User-Agent, статистики координат и перемещений курсора.
- Пропуски категорий заменяются на `missing`, числовых агрегатов — на 0. Доли заполненности дополнительно сохраняют информацию об отсутствии координат.
- Отдельная модель использует события после удаления строк, совпадающих во всех исходных полях.
- Наличие автоматизированного User-Agent не задаёт метку напрямую: все зависимости учатся по train.

### Проверка по времени

Обучение всегда предшествует проверочному периоду. Отрезки 11–13, 14–16 и 17–19 апреля использовались для выбора признаков и настроек. Поэтому их оценки могут быть оптимистичны после многократных экспериментов; скрытый тест остаётся отдельной проверкой. Абсолютные даты окна и тестовые метки не используются как признаки.

## 1. Окружение

Установите зависимости: `python -m pip install -r requirements-notebook.txt`. Запускайте ноутбук из корня репозитория. Проверенное окружение — Python 3.13.13, CPU. Обучение занимает несколько минут; обращения к внешним API отсутствуют.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
from bot_detection.model import CONFIG, feature_sets, ensemble_predict, validate

DATA = Path("data")
RUN_VALIDATION = False  # True: дополнительно переобучить модели на всех трёх временных отрезках.
manifest = json.loads(Path("artifacts/manifest.json").read_text(encoding="utf-8"))
print("Python:", platform.python_version())
print({name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scikit-learn", "catboost"]})

Python: 3.13.13
{'numpy': '2.4.6', 'pandas': '3.0.5', 'scikit-learn': '1.9.1', 'catboost': '1.2.10'}


## 2. Данные и проверка входа

Нужны три файла из исходного архива задания. Проверяем их контрольные суммы, уникальность куки и отсутствие пересечения train и test. В репозиторий не включаются исходные журналы; у проверяющего они уже есть.

In [2]:
for name, expected in manifest["inputs_sha256"].items():
    actual = hashlib.sha256((DATA / name).read_bytes()).hexdigest()
    assert actual == expected, f"Другой набор данных: {name}"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
events = pd.read_csv(DATA / "events.csv.gz", parse_dates=["event_ts"])
meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
assert meta.cookie_id.is_unique
assert train.target.isin([0, 1]).all()
display(pd.DataFrame({"Набор": ["train", "test", "events"], "Строк": [len(train), len(test), len(events)]}))
print("Доля положительного класса в train:", round(train.target.mean(), 5))
print("Полностью повторяющихся строк событий:", int(events.duplicated().sum()))

,Набор,Строк
0,train,11091
1,test,4909
2,events,328905


Доля положительного класса в train: 0.08106
Полностью повторяющихся строк событий: 4863


## 3. Построение признаков

Код агрегирования находится в `bot_detection/features.py`. Он сортирует события внутри куки по времени, рассчитывает паузы и движения курсора, затем агрегирует до одной строки на cookie_id. Каждый из четырёх наборов сохраняет исходный порядок train, затем test.

In [3]:
frames = feature_sets(meta, events)
display(pd.DataFrame({"Группа": list(frames), "Число признаков": [f.shape[1] for f in frames.values()]}))

Calculated features from recorded events


Calculated features after exact duplicate removal


,Группа,Число признаков
0,base,156
1,pointer,166
2,deduplicated,166
3,device,189


## 4. Валидация и baseline

Метрика перебирает пороги с Recall ≥ 0,70 и берёт максимальный Precision; одинаковые score обрабатываются вместе. Baseline — Random Forest на числе событий и уникальных объявлений. Ниже показаны сохранённые результаты экспериментов. Установите `RUN_VALIDATION=True` выше, чтобы пересчитать их. Три строки приведены таблицей для точного сравнения, без усреднения разных метрик.

In [4]:
if RUN_VALIDATION:
    validate(frames, train, Path("validation.csv"))
    validation = pd.read_csv("validation.csv")
else:
    validation = pd.read_csv("artifacts/validation.csv")
display(validation[["period", "positive_cookies", "baseline_precision_at_recall_0_70", "precision_at_recall_0_70", "average_precision"]].round(4))
print("Средний Precision при Recall >= 70%:", round(validation.precision_at_recall_0_70.mean(), 4))

,period,positive_cookies,baseline_precision_at_recall_0_70,precision_at_recall_0_70,average_precision
0,early,199,0.0964,0.6731,0.7562
1,middle,195,0.1050,0.7709,0.8029
2,late,160,0.1062,0.7943,0.7883


Средний Precision при Recall >= 70%: 0.7461


## 5. Обучение на всей размеченной выборке

Смешиваются вероятности: основные признаки — 12,5%, с перемещениями курсора — 37,5%, после удаления дубликатов — 25%, расширенные признаки курсора и устройства — 25%. Все модели имеют глубину 6, learning_rate=0,045, l2_leaf_reg=5, random_seed=42. Обучение включает 700 деревьев; в трёх компонентах используются первые 500, в основной — все 700. Число CPU-потоков зафиксировано в CONFIG. Настройки совпадают с теми, которыми получен отправленный файл.

In [5]:
n_train = len(train)
score = ensemble_predict(frames, train.target.to_numpy(), np.arange(n_train), np.arange(n_train, len(meta)))
submission = pd.DataFrame({"cookie_id": test.cookie_id, "score": score})
assert submission.cookie_id.is_unique and len(submission) == len(test)
assert submission.cookie_id.equals(test.cookie_id)
assert np.isfinite(score).all() and submission.score.between(0, 1).all()
submission.to_csv("submission.csv", index=False, lineterminator="\r\n")
print("Записан submission.csv:", len(submission), "строк")

Trained base


Trained pointer


Trained deduplicated


Trained device


Записан submission.csv: 4909 строк


## 6. Совпадение с отправленным решением

Контрольная сумма проверяет весь файл, включая порядок куки и значения score. Она используется только после предсказания для проверки воспроизводимости и не участвует в построении признаков или обучении.

In [6]:
actual_hash = hashlib.sha256(Path("submission.csv").read_bytes()).hexdigest()
assert actual_hash == manifest["submission_sha256"], "CSV отличается от эталонного файла"
print("SHA-256:", actual_hash)
print("Полное побайтовое совпадение с artifacts/submission.csv подтверждено.")

SHA-256: 45c6660d9e40560903bdf7bd0d649c1e294a2c9ab80cfc86401ce34874ac8a61
Полное побайтовое совпадение с artifacts/submission.csv подтверждено.


## Выводы

Локальные Precision при Recall ≥ 70%: **0,6731 / 0,7709 / 0,7943**. Ансамбль заметно превосходит baseline по двум счётчикам, но не даёт идеального разделения классов. Проверялись CatBoost с разными настройками, LightGBM, ExtraTrees, HistGradientBoosting, отбор признаков, признаки последовательности и времени, удаление дубликатов и смеси моделей. Выбор опирался на временную валидацию; баллы на скрытом тесте не использовались для восстановления отдельных меток.

Для сдачи: `artifacts/submission.csv`, код репозитория и `description.txt`. Для запуска без Jupyter: `python train.py --data data --output submission.csv`.